# Air+ : pipeline d'analyse

Ce notebook exécute **le même moteur Python** que le dashboard. Il prend les trois Excel MES, PLM et ERP, refait le nettoyage, les rapprochements et les calculs, puis produit le ZIP d'analyse à ouvrir dans le dashboard (menu Données).

Aucune donnée n'est simulée. Le ZIP ne contient que des identités pseudonymisées.


## 1. Installer le moteur embarqué

Le code ci-dessous est l'archive exacte de `airplus/engine`, vérifiée par son empreinte.


In [ ]:
import base64, hashlib, importlib.util, io, os, subprocess, sys, zipfile
if importlib.util.find_spec("openpyxl") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openpyxl"])
ENGINE_B64 = """
...
Lmpzb25QSwUGAAAAAA8ADwD6BAAAw3oEAAAA
"""
raw = base64.b64decode(ENGINE_B64)
assert hashlib.sha256(raw).hexdigest() == "1700f245bf68b9fee7440bee9c89adf692b68009040c8ab612ae4c5dcec03ca7", "Moteur embarqué altéré"
WORK = os.path.abspath("airplus_colab")
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    z.extractall(WORK)
sys.path.insert(0, WORK)
from airplus import engine
print("Moteur Air+", engine.ENGINE_VERSION)


## 2. Charger les trois Excel

Dans Colab, sélectionnez `MES_Extraction_FR.xlsx`, `PLM_DataSet_FR.xlsx` et `ERP_Equipes Airplus_FR.xlsx`. Hors Colab, définissez la variable `AIRPLUS_SOURCES` vers leur dossier.


In [ ]:
def _pick(names, key):
    hits = [n for n in names if key in n.upper()]
    assert hits, f"Fichier {key} introuvable parmi {list(names)}"
    return sorted(hits, key=len)[0]

try:
    from google.colab import files as _colab_files
    uploaded = _colab_files.upload()
except ImportError:
    folder = os.environ["AIRPLUS_SOURCES"]
    uploaded = {f: open(os.path.join(folder, f), "rb").read() for f in os.listdir(folder) if f.lower().endswith(".xlsx")}
names = {k: _pick(uploaded, k.upper()) for k in ("mes", "plm", "erp")}
print(names)


## 3. Exécuter le moteur


In [ ]:
zip_bytes, analysis, manifest, identity = engine.run(uploaded[names["mes"]], uploaded[names["plm"]], uploaded[names["erp"]], names)
del identity  # les identites restent en memoire le temps du controle, jamais dans le ZIP
print("Empreinte du contenu :", manifest["content_sha256"])
print("Pseudonymisation :", manifest["pseudonymisation"]["fuites"] or "aucune fuite", "sur", manifest["pseudonymisation"]["fichiers_controles"], "fichiers")


## 4. Indicateurs et contrôles


In [ ]:
for k in analysis["kpis"]:
    print(f'{k["libelle"]:<58} {str(k["affichage"]):>16} {k["unite"] if k["unite"] != "s" else "":<14} n={k["effectif"]}')
print("\nSupply chain :", analysis["supply"]["n_fournisseurs"], "fournisseurs,", analysis["supply"]["valeur_totale_eur"], "EUR, top 3 :", analysis["supply"]["part_top3_pct"], "%")
print("\nJointures :")
for j in analysis["joins"]:
    print(f' - {j["jointure"]:<22} {j["numerateur"]}/{j["denominateur"]} {j["unite"]} ({j["couverture_pct"]} %)')
print("\nGoulots candidats :")
for c in analysis["candidates"][:5]:
    print(f' - poste {c["poste"]:>2} {c["intitule"]} : écart {c["ecart_s"]} s, {c["src"]}. {c["statut"]}')


## 5. Écrire et télécharger le ZIP

Comparez l'empreinte à celle de l'analyse préchargée : deux empreintes égales signifient des tables strictement identiques.


In [ ]:
EXPECTED_PRELOADED = "2f1a3a60cfba50106567c61af821197fe9645b2d7fc426c4339cd35b11120b8e"
OUT = "airplus_analyse.zip"
with open(OUT, "wb") as f:
    f.write(zip_bytes)
print("Identique à l'analyse préchargée :", manifest["content_sha256"] == EXPECTED_PRELOADED)
try:
    from google.colab import files as _colab_files
    _colab_files.download(OUT)
except ImportError:
    print("ZIP écrit :", os.path.abspath(OUT))
